# ResNet-18 on an 8 x 8 systolic array

Choose one of the bundled pictures or upload your own. The board preprocesses
it, runs it through the 8 x 8 systolic array, and prints the top-5 ImageNet
labels with a CORRECT or INCORRECT verdict.

This notebook is a demonstration. It shows the overlay identity and the
physical job count, so you can see the work ran on the fabric, but it performs
no host/board digest comparison and writes no acceptance evidence. For
deterministic numerical acceptance against the host record, use
`run_on_board.py`, which runs the synthetic regression tensor on the board and
writes `board-evidence.json`.

One forward pass is 1,814,073,344 MACs on 64 processing elements and takes
roughly an hour. Start it and let it run.

## 1. Connect to the deployed release

In [ ]:
import json
import os
from pathlib import Path
import sys
import time

import numpy as np

if not sys.platform.startswith('linux'):
    raise RuntimeError('Open this notebook on the PYNQ-Z1, not Windows')

start = Path.cwd().resolve()
release_root = next(
    (candidate for candidate in (start, *start.parents)
     if (candidate / 'deployment.json').is_file()
     and (candidate / 'examples/resnet18/model').is_dir()
     and (candidate / 'build/vivado/npu_matrix_8x8/artifacts').is_dir()),
    None,
)
if release_root is None:
    raise RuntimeError('Open this notebook from a deployed ResNet-18 release')

sys.path.insert(0, str(release_root))
model_dir = release_root / 'examples/resnet18/model'
artifact_dir = release_root / 'build/vivado/npu_matrix_8x8/artifacts'
print(f'release root : {release_root}')
print(f'python       : {sys.executable}')

## 2. Program the FPGA

This loads the bitstream and reports the accelerator the notebook is actually
talking to. An 8 x 8 array with 64 processing elements here means the work
below runs on the fabric, not on the ARM cores.

In [ ]:
os.environ['XILINX_XRT'] = '/usr'
from src.runtime import (
    NPUModelRuntime,
    NPURuntime,
    load_model_package,
    load_pynq_runtime,
)

physical = load_pynq_runtime(artifact_dir / 'npu_matrix.bit')
assert isinstance(physical, NPURuntime), type(physical)
assert (physical.max_m, physical.max_n, physical.max_k) == (8, 8, 256), \
    'Expected 8 x 8 hardware with MAX_K=256'

model = load_model_package(model_dir / 'resnet18.npu.json')
manifest = json.loads(
    (model_dir / 'resnet18.npu.json').read_text(encoding='utf-8')
)

print(f'runtime class       : {type(physical).__name__}')
print(f'systolic array      : {physical.max_m} x {physical.max_n}, MAX_K={physical.max_k}')
print(f'processing elements : {physical.max_m * physical.max_n}')
print(f'bitstream           : {artifact_dir / "npu_matrix.bit"}')

## 3. Choose a picture

Pick one of the bundled pictures from the dropdown, or upload your own. An
upload always wins over the dropdown. The bundled set is five Creative Commons
photographs fetched by `scripts/download_gallery.py`; each one declares the
ImageNet class it should produce, so Step 6 can say whether the network got it
right.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

gallery_path = release_root / 'examples/resnet18/gallery-source.json'
gallery = []
if gallery_path.is_file():
    for entry in json.loads(gallery_path.read_text(encoding='utf-8'))['images']:
        candidate = model_dir / entry['filename']
        if candidate.is_file():
            gallery.append((entry['expected_class'], candidate, entry))

options = [
    (f"{expected}   ({path.name})", index)
    for index, (expected, path, _entry) in enumerate(gallery)
]
options.append(('-- upload my own picture --', -1))

chooser = widgets.Dropdown(
    options=options,
    value=0 if gallery else -1,
    description='picture:',
    layout=widgets.Layout(width='460px'),
)
uploader = widgets.FileUpload(accept='image/*', multiple=False)
display(chooser, uploader)

print(f'{len(gallery)} bundled pictures available.')
if not gallery:
    print('Run scripts/download_gallery.py to fetch them, or upload your own.')
print('An upload overrides the dropdown. Then run the next cell.')

## 4. Preprocess on the board, and look at it before any inference

The left panel is the picture. The right panel is the exact signed-INT8 tensor
the NPU will receive, dequantized back to pixels. Confirm both show the same
subject: a wrong crop or a swapped channel order is visible here, before an
hour of compute.

In [ ]:
from src.export.imagenet import (
    dequantized_preview,
    input_scale,
    load_rgb_image,
    preprocess_image,
)
import matplotlib.pyplot as plt


def uploaded_file(widget):
    """Return (name, bytes) for ipywidgets 7 or 8, or None when nothing was picked."""
    value = widget.value
    if not value:
        return None
    if isinstance(value, dict):
        name, payload = next(iter(value.items()))
        return name, bytes(payload['content'])
    entry = value[0]
    return entry['name'], bytes(entry['content'])


picked = uploaded_file(uploader)
if picked is not None:
    name, content = picked
    image_path = Path('/tmp') / f'npu-demo-{name}'
    image_path.write_bytes(content)
    expected_class = None
    image_label = f'{name}   (uploaded, {len(content):,} bytes)'
elif chooser.value >= 0:
    expected_class, image_path, entry = gallery[chooser.value]
    image_label = (
        f"{image_path.name}   {entry['license']['spdx']}, "
        f"{entry['license']['attribution']}"
    )
else:
    image_path = model_dir / 'demo-image.jpg'
    expected_class = 'Samoyed'
    image_label = f'{image_path.name}   (bundled sample)'

scale = input_scale(manifest, 'input')
logit_scale = input_scale(manifest, 'logits')

original = load_rgb_image(image_path)
_crop, npu_input = preprocess_image(original, scale)
preview = dequantized_preview(npu_input, scale)

figure, axes = plt.subplots(1, 2, figsize=(9, 5), facecolor='#fcfcfb')
axes[0].imshow(original)
axes[0].set_title(
    f'the picture   {original.shape[1]} x {original.shape[0]}', color='#0b0b0b'
)
axes[1].imshow(preview)
axes[1].set_title('exactly what the NPU receives   224 x 224 INT8', color='#0b0b0b')
for axis in axes:
    axis.set_xticks([])
    axis.set_yticks([])
figure.suptitle(image_label, color='#52514e', fontsize=10)
figure.tight_layout()
plt.show()

print(f'input tensor  : {npu_input.shape} {npu_input.dtype}')
print(f'input scale   : {scale:.9f}')
print(f'expected class: {expected_class or "not declared; judge it yourself"}')

## 5. Run it on the NPU

`physical jobs` counts matrix operations the fabric actually executed. A value
above zero is what separates this from a host simulation.

In [ ]:
started = time.monotonic()
result = NPUModelRuntime(physical, model).run(
    {'input': npu_input}, software_timeout=86400.0
)
elapsed = time.monotonic() - started

assert result.metrics.physical_jobs > 0, 'no physical job ran; this was not the board'
print(f'elapsed       : {elapsed / 60:.1f} min')
print(f'MACs          : {result.metrics.mac_count:,}')
print(f'physical jobs : {result.metrics.physical_jobs:,}')

## 6. Read the prediction

The top-5 ImageNet classes, highest first. For a bundled picture the expected
class is known, so the notebook states `CORRECT` or `INCORRECT`. For your own
upload there is no ground truth: look at the picture in Step 4 and judge it.

In [ ]:
from src.export.imagenet import decode_logits, load_class_names

class_names = load_class_names(model_dir / 'imagenet-classes.txt')
top5 = decode_logits(result.outputs['logits'], logit_scale, class_names, top_k=5)

for rank, prediction in enumerate(top5, start=1):
    print(f'{rank}. {prediction.name:<34} {prediction.probability:7.2%}')
print()
print(f'predicted: {top5[0].name}   (index {top5[0].index})')
if expected_class is None:
    print('expected:  not declared for an uploaded picture; judge it yourself')
else:
    verdict = 'CORRECT' if top5[0].name == expected_class else 'INCORRECT'
    print(f'expected:  {expected_class}')
    print(f'verdict:   {verdict}')

labels = [prediction.name for prediction in reversed(top5)]
values = [prediction.probability for prediction in reversed(top5)]
colors = ['#86b6ef'] * (len(values) - 1) + ['#2a78d6']

figure, axis = plt.subplots(figsize=(8, 3.2), facecolor='#fcfcfb')
axis.set_facecolor('#fcfcfb')
bars = axis.barh(labels, values, color=colors, height=0.62)
for bar, value in zip(bars, values):
    axis.text(
        value + max(values) * 0.02,
        bar.get_y() + bar.get_height() / 2,
        f'{value:.1%}',
        va='center',
        color='#52514e',
        fontsize=10,
    )
axis.set_xlim(0, max(values) * 1.22)
axis.set_title('top-5 ImageNet prediction', color='#0b0b0b', loc='left', pad=12)
axis.tick_params(colors='#52514e', length=0)
axis.xaxis.set_visible(False)
for side in ('top', 'right', 'bottom'):
    axis.spines[side].set_visible(False)
axis.spines['left'].set_color('#d8d7d2')
figure.tight_layout()
plt.show()